# Preparación de datos y entrenamiento de modelo en SageMaker

En este notebook se implementa un flujo completo para el dataset del Titanic:

1. Lectura del dataset crudo.
2. Limpieza y feature engineering mediante un SageMaker Processing Job.
3. Generación de datasets de entrenamiento, validación y test.
4. Entrenamiento de un modelo con el algoritmo built-in XGBoost de SageMaker.

El objetivo del modelo es predecir la variable `Survived`.

## 1. Configuración inicial de SageMaker

Primero se configura la sesión de SageMaker, el bucket por defecto y el rol IAM que usará SageMaker para lanzar los jobs.

In [ ]:
import os
import boto3
import sagemaker

from sagemaker import get_execution_role

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = get_execution_role()
region = sess.boto_region_name

prefix = "sagemaker/titanic-lab"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)

## 2. Comprobación del dataset crudo

El fichero de entrada será el CSV original del Titanic. Este fichero todavía contiene columnas sin limpiar, valores nulos y variables categóricas.

In [ ]:
import pandas as pd

local_raw_data = "titanic/titanic-dataset.csv"

df = pd.read_csv(local_raw_data)
df.head()

In [ ]:
df.info()

## 3. Script de preprocesamiento externo

El preprocesamiento se encuentra en el archivo `processing_titanic.py`.

Este script será ejecutado por un SageMaker Processing Job. El notebook no contiene la lógica de limpieza directamente, sino que invoca ese script externo, igual que en los ejemplos de clase.

In [ ]:
import os

script_path = "processing_titanic.py"

print("Ruta absoluta:", os.path.abspath(script_path))
print("Existe:", os.path.exists(script_path))

## 4. Lanzamiento del Processing Job

Se crea un `SKLearnProcessor` para ejecutar el script de preprocesamiento en SageMaker.

El job recibe como entrada el dataset crudo del Titanic y genera tres salidas:

- entrenamiento
- validación
- test

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="titanic-processing"
)

sklearn_processor.run(
    code="processing_titanic.py",
    inputs=[
        ProcessingInput(
            source=local_raw_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/{prefix}/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"s3://{bucket}/{prefix}/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/{prefix}/test"
        )
    ]
)

## 5. Rutas S3 generadas

El Processing Job ha dejado los datasets procesados en S3. Estas rutas se usarán como entrada del entrenamiento.

In [ ]:
train_s3_uri = f"s3://{bucket}/{prefix}/train"
validation_s3_uri = f"s3://{bucket}/{prefix}/validation"
test_s3_uri = f"s3://{bucket}/{prefix}/test"

train_file_s3_uri = f"{train_s3_uri}/train.csv"
validation_file_s3_uri = f"{validation_s3_uri}/validation.csv"
test_file_s3_uri = f"{test_s3_uri}/test.csv"

print("Train:", train_file_s3_uri)
print("Validation:", validation_file_s3_uri)
print("Test:", test_file_s3_uri)

## 6. Entrenamiento con XGBoost built-in

Se usará el algoritmo built-in XGBoost de SageMaker.

Como el problema es de clasificación binaria, se configura el objetivo `binary:logistic`.

El dataset ya está en formato CSV sin cabecera y con `Survived` como primera columna.

In [ ]:
from sagemaker.inputs import TrainingInput

xgboost_container = sagemaker.image_uris.retrieve(
    "xgboost",
    region,
    "1.7-1"
)

print(xgboost_container)

In [ ]:
s3_input_train = TrainingInput(
    s3_data=train_s3_uri,
    content_type="csv"
)

s3_input_validation = TrainingInput(
    s3_data=validation_s3_uri,
    content_type="csv"
)

In [ ]:
xgb = sagemaker.estimator.Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/model-output",
    sagemaker_session=sess,
    base_job_name="titanic-xgboost"
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    max_depth=4,
    eta=0.2,
    gamma=2,
    min_child_weight=1,
    subsample=0.8,
    num_round=100
)

## 7. Ejecución del Training Job

Se lanza el entrenamiento usando los canales `train` y `validation`.

En los logs del entrenamiento aparecerá la métrica `validation:auc`.

In [ ]:
xgb.fit(
    {
        "train": s3_input_train,
        "validation": s3_input_validation
    }
)

## 8. Artefacto del modelo entrenado

Cuando termina el entrenamiento, SageMaker guarda el modelo en S3 como un artefacto `model.tar.gz`.

In [ ]:
model_artifact = xgb.model_data
print("Modelo guardado en:", model_artifact)

## 9. Despliegue temporal del modelo

Esta parte es opcional. Se despliega un endpoint temporal para hacer predicciones.

Después de probarlo, se debe eliminar el endpoint para evitar costes.

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import StringDeserializer

predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=StringDeserializer()
)

## 10. Descarga del dataset de test

Se descarga el conjunto de test procesado para probar algunas predicciones.

In [ ]:
test_local_path = "test.csv"

s3_client = boto3.client("s3")
test_key = f"{prefix}/test/test.csv"

s3_client.download_file(bucket, test_key, test_local_path)

test_data = pd.read_csv(test_local_path, header=None)
test_data.head()

In [ ]:
import numpy as np

y_test = test_data.iloc[:, 0]
X_test = test_data.iloc[:, 1:]

sample = X_test.head(10)

raw_predictions = predictor.predict(sample.to_numpy())
raw_predictions

In [ ]:
predicted_probabilities = np.array(
    [float(value) for value in raw_predictions.strip().split("\n")]
)

predicted_classes = (predicted_probabilities >= 0.5).astype(int)

pd.DataFrame({
    "real": y_test.head(10).values,
    "probabilidad_supervivencia": predicted_probabilities,
    "prediccion": predicted_classes
})

## 11. Eliminación del endpoint

Se elimina el endpoint para evitar costes innecesarios.

In [ ]:
predictor.delete_endpoint()